# Vision EDA — car damage dataset (Sprint 1)

**Owner:** Wiem · **Agent:** Vision · **Dataset:** Roboflow Universe `sindhu/car_dent_scratch_detection-sekha` v1 (CC BY 4.0)

Goals of this notebook:
1. Check the leak-free split produced by `agents/vision/split_dataset.py`.
2. Understand class balance, objects per image, box sizes and image resolutions.
3. Check the mapping `class → (part, damage_type)` used to fill `Damage` (schema section 4).
4. Write down the decisions for the YOLOv8n baseline.

> Run from the repo root venv. Clear outputs before committing (`jupyter nbconvert --clear-output --inplace notebooks/vision/01_eda.ipynb`).

In [ ]:
import importlib.util
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from PIL import Image, ImageDraw

# Locate the repo root (folder containing .git) so paths work from any cwd.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
CFG = yaml.safe_load((ROOT / "agents/vision/config.yaml").read_text(encoding="utf-8"))
DATA = ROOT / CFG["split"]["output_dir"]
MAPPING = yaml.safe_load((ROOT / CFG["class_mapping"]).read_text(encoding="utf-8"))
NAMES = yaml.safe_load((DATA / "data.yaml").read_text(encoding="utf-8"))["names"]
SPLITS = ["train", "valid", "test"]
print(ROOT, DATA, len(NAMES), "classes", sep="\n")

## 1. Load every label into one table

In [ ]:
rows, images = [], []
for split in SPLITS:
    for img in sorted((DATA / split / "images").iterdir()):
        label = DATA / split / "labels" / f"{img.stem}.txt"
        lines = label.read_text(encoding="utf-8").split("\n") if label.exists() else []
        lines = [line for line in lines if line.strip()]
        images.append({"split": split, "image": img.name, "n_objects": len(lines)})
        for line in lines:
            cls, xc, yc, w, h = line.split()[:5]
            name = NAMES[int(cls)]
            rows.append({
                "split": split, "image": img.name, "class": name,
                "xc": float(xc), "yc": float(yc), "w": float(w), "h": float(h),
                **MAPPING[name],
            })

boxes = pd.DataFrame(rows)
imgs = pd.DataFrame(images)
boxes["area"] = boxes["w"] * boxes["h"]  # relative area (0..1)
print(f"{len(imgs)} images, {len(boxes)} boxes")
imgs.groupby("split").agg(images=("image", "count"), boxes=("n_objects", "sum"),
                          empty_images=("n_objects", lambda s: int((s == 0).sum())))

## 2. Class distribution per split

In [ ]:
counts = boxes.pivot_table(index="class", columns="split", values="image",
                           aggfunc="count", fill_value=0)[SPLITS]
counts = counts.sort_values("train")
counts.plot.barh(figsize=(9, 7), title="Boxes per class and split")
plt.xlabel("boxes")
plt.tight_layout()
plt.show()
counts[(counts["valid"] == 0) | (counts["test"] == 0)]  # classes missing from valid/test

## 3. What the agent will output: `part` and `damage_type`

These are the values that will fill `Damage.part` / `Damage.damage_type`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["part", "damage_type", "severity_hint"], strict=True):
    boxes[col].fillna("none").value_counts().plot.bar(ax=ax, title=col)
plt.tight_layout()
plt.show()
print("Share of boxes with part == 'unknown':", round((boxes["part"] == "unknown").mean(), 3))

## 4. Objects per image and box sizes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
imgs["n_objects"].plot.hist(ax=axes[0], bins=range(imgs["n_objects"].max() + 2),
                            title="Objects per image")
boxes["area"].plot.hist(ax=axes[1], bins=40, title="Box area (fraction of image)")
boxes.plot.scatter(x="w", y="h", s=3, alpha=0.3, ax=axes[2], title="Box width vs height")
plt.tight_layout()
plt.show()
small = (boxes["area"] < 0.01).mean()
print(f"Boxes smaller than 1% of the image: {small:.1%}")
boxes.groupby("class")["area"].median().sort_values().round(3)

## 5. Image resolutions

YOLO resizes to `imgsz`; this tells us whether 640 is reasonable.

In [ ]:
sizes = Counter()
for split in SPLITS:
    for img in (DATA / split / "images").iterdir():
        with Image.open(img) as im:
            sizes[im.size] += 1
pd.Series(sizes).sort_values(ascending=False).head(10).rename("images")

## 6. Visual check of the mapping

Check the ambiguous classes: is `backlight` a rear window or a tail light? What do `dent` and `Door-outer-damage` look like?

In [ ]:
def show_examples(class_name: str, n: int = 4, split: str = "train") -> None:
    sample = boxes[(boxes["class"] == class_name) & (boxes["split"] == split)]
    sample = sample.drop_duplicates("image").head(n)
    if sample.empty:
        print(f"No '{class_name}' in {split}")
        return
    _, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 4), squeeze=False)
    for ax, (_, row) in zip(axes[0], sample.iterrows(), strict=False):
        im = Image.open(DATA / split / "images" / row["image"]).convert("RGB")
        draw = ImageDraw.Draw(im)
        width, height = im.size
        img_boxes = boxes[(boxes["image"] == row["image"]) & (boxes["split"] == split)]
        for _, b in img_boxes.iterrows():
            x1, y1 = (b["xc"] - b["w"] / 2) * width, (b["yc"] - b["h"] / 2) * height
            x2, y2 = (b["xc"] + b["w"] / 2) * width, (b["yc"] + b["h"] / 2) * height
            draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
            draw.text((x1 + 3, y1 + 3), b["class"], fill="yellow")
        ax.imshow(im)
        ax.set_title(f"{class_name}\n-> {MAPPING[class_name]['part']} / "
                     f"{MAPPING[class_name]['damage_type']}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


for name in ["backlight", "dent", "Door-outer-damage", "headlight-damage"]:
    show_examples(name)

## 7. Leakage check

No source photo (augmented copies and `- Copy` duplicates included) may appear in two splits.

In [ ]:
# Load group_key() from the split script without touching sys.path.
spec = importlib.util.spec_from_file_location("split_dataset", ROOT / "agents/vision/split_dataset.py")
split_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(split_module)
group_key = split_module.group_key

groups = {s: {group_key(n) for n in imgs.loc[imgs["split"] == s, "image"]} for s in SPLITS}
for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]:
    print(f"{a} ∩ {b}: {len(groups[a] & groups[b])} shared source photos")
assert not (groups["train"] & groups["test"]), "Leakage between train and test!"


## 8. Findings and decisions for the baseline

- **Data:** 719 source photos, ~3 augmented copies each (Roboflow) → 1674 / 366 / 366 images and 2307 / 576 / 507 boxes (train / valid / test). Split by source photo, seed 42. No empty images (~1.4 damages per image).
- **Leakage:** 0 shared source photos between any two splits (section 7).
- **Rare classes:** `Major-door-outer-damage` (18 boxes) and `Minor-door-outer-damage` (3 boxes) are absent from valid/test. Kept for the sprint-1 baseline (mAP50 is computed on classes present in the split); to merge into `Door-outer-damage` in week 2 (same `part`/`damage_type`).
- **Mapping:** 26.2% of boxes have `part = unknown` (generic `dent` / `scratch` classes). Impact for the NLP agent: no part to compare for those damages. Visual check: `backlight` = tail light → mapped to `tail_light` / `crack`.
- **Box sizes:** only 4.6% of boxes are smaller than 1% of the image → `imgsz=640` is enough.
- **Limitation:** no "no damage" images → the agent must handle the empty-detection case explicitly (week 3 edge cases).
- **Severity:** Major/Medium/Minor prefixes give a `severity_hint` for most classes → basis for the week-2 severity rule.
- **Baseline plan:** YOLOv8n, `imgsz=640`, CPU; metric = mAP50 on `test`.